<a href="https://colab.research.google.com/github/losos-1/lab2-4/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 6
## Кривые и поверхности Безье в задаче построения букв русского алфавита

### Цель работы
Освоить построение кривых Безье и их использование для задания шрифтовых контуров. Реализовать вычисление точки кривой, алгоритм де Кастельжо, разбиение кривой, а затем применить эти идеи к векторным контурам букв русского алфавита.

Дополнительно необходимо показать, как 2D-контур буквы может быть превращён в 3D-форму с помощью боковых поверхностей, которые можно интерпретировать как поверхности Безье.

### Центральная задача
Каждый студент получает **5 букв русского алфавита**, определяемых по его номеру ИСУ. Для этих букв нужно построить контуры, стилевые варианты, исследовать изменение кривизны и построить 3D-визуализацию одной выбранной буквы.

### Рекомендуемые инструменты
- `numpy` — вычисления;
- `matplotlib` — 2D-графики и проверочные визуализации;
- `plotly` — **рекомендуется для 3D-визуализации**, потому что интерактивный график можно вращать и лучше рассматривать форму буквы;
- `matplotlib.textpath.TextPath` — получение векторных контуров букв из шрифта.



## 1) Индивидуальный вариант по номеру ИСУ

Набор из пяти букв строится детерминированно по шестизначному номеру ИСУ.

Используется русский алфавит:

```text
А Б В Г Д Е Ё Ж З И Й К Л М Н О П Р С Т У Ф Х Ц Ч Ш Щ Ъ Ы Ь Э Ю Я
```

Для каждой буквы вычисляется псевдослучайный вес на основе пары `(ИСУ, буква)`, после чего выбираются 5 букв с наименьшими весами. Такой способ даёт случайно выглядящий, но полностью воспроизводимый вариант.

### Пример выданных вариантов
В ячейке ниже показаны варианты для нескольких номеров ИСУ.


In [1]:
import math
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from matplotlib.path import Path

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

RUS_ALPHABET = 'АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'

def letters_by_isu(isu, k=5, alphabet=RUS_ALPHABET):
    isu = str(isu)
    scored = []
    for letter in alphabet:
        raw = f'{isu}-{letter}'.encode('utf-8')
        digest = hashlib.sha256(raw).digest()
        score = int.from_bytes(digest[:8], byteorder='big')
        scored.append((score, letter))
    return [letter for _, letter in sorted(scored)[:k]]

ISU = 504851
LETTERS = letters_by_isu(ISU)

print('ИСУ:', ISU)
print('Индивидуальные буквы:', ' '.join(LETTERS))

FONT = FontProperties(family='DejaVu Sans')

ИСУ: 504851
Индивидуальные буквы: К И Ж Е Щ



### Что нужно сделать для своего набора букв

Для пяти букв, полученных по ИСУ, необходимо:

1. построить 2D-контуры;
2. показать контрольные точки / контрольные многоугольники хотя бы для части криволинейных сегментов;
3. построить четыре варианта каждой буквы:
   - `regular` — исходный;
   - `italic` — курсив через аффинный сдвиг;
   - `bold` — приближённая жирность через геометрическое расширение;
   - `custom` — авторская деформация;
4. исследовать изменение кривизны на одной выбранной букве;
5. построить 3D-визуализацию одной буквы. Для 3D рекомендуется использовать **Plotly**.



## 2) Базовые функции для кривых Безье

Реализуем линейную интерполяцию, полиномы Бернштейна, вычисление точки кривой по формуле и алгоритм де Кастельжо.


In [ ]:

# TODO: реализуйте базовые функции

def lerp(P, Q, t):
    raise NotImplementedError


def bernstein(n, i, t):
    raise NotImplementedError


def bezier_point_bernstein(points, t):
    raise NotImplementedError


def de_casteljau(points, t):
    raise NotImplementedError


def bezier_curve(points, samples=200):
    raise NotImplementedError


def split_bezier(points, t):
    raise NotImplementedError



## 3) Минимальный self-check

Проверим, что вычисление точки через полиномы Бернштейна и через алгоритм де Кастельжо совпадает.


In [ ]:

segment = np.array([
    [0.0, 0.0],
    [1.2, 3.0],
    [4.2, 3.0],
    [5.2, 0.0]
])

t = 0.37
p1 = bezier_point_bernstein(segment, t)
p2, _ = de_casteljau(segment, t)

print('Точка по Бернштейну :', np.round(p1, 6))
print('Точка де Кастельжо  :', np.round(p2, 6))
print('Норма разности      :', np.linalg.norm(p1 - p2))
assert np.allclose(p1, p2)



## 4) Базовые визуализации: Бернштейн, де Кастельжо, разбиение

Ниже нужно построить три базовые визуализации:

1. полиномы Бернштейна для кубической кривой;
2. геометрическая схема алгоритма де Кастельжо;
3. разбиение кубической кривой при заданном параметре.


In [ ]:
# Полиномы Бернштейна степени 3


# Де Кастельжо


# Разбиение



## 5) Получение векторных контуров букв

Векторные шрифты хранят буквы как контуры из отрезков и кривых. В этой работе все сегменты приводятся к кубическим кривым Безье:

- отрезок записывается как кубическая кривая с четырьмя коллинеарными контрольными точками;
- квадратичная кривая переводится в кубическую;
- кубическая кривая используется напрямую.

Это удобно, потому что дальше все буквы можно обрабатывать единой функцией.


In [ ]:

# TODO: реализуйте преобразование сегментов шрифта к кубическим кривым Безье
# Рекомендуется привести отрезки и квадратичные кривые к кубическому виду.

def line_to_cubic(P0, P1):
    raise NotImplementedError


def quadratic_to_cubic(P0, Q, P2):
    raise NotImplementedError


def raw_letter_segments(letter, size=1.0, font_prop=FONT):
    raise NotImplementedError


def get_letter_segments(letter, target_height=1.0):
    raise NotImplementedError



## 6) Визуализация пяти исходных букв

Для каждой буквы строятся контуры, полученные из шрифта. На этом этапе это вариант `regular`.


In [ ]:
# TO DO: визуализируйте кривые Безье для своих букв


## 7) Контрольные многоугольники для выбранных букв

Для читаемости показываются не все контрольные многоугольники, а первые несколько сегментов. Этого достаточно, чтобы увидеть, что форма буквы задаётся контрольными точками и составными кривыми.


In [ ]:
# TO DO


## 8) Стилевые преобразования: regular, italic, bold, custom

В этой работе стиль рассматривается как геометрическое преобразование контура.

### Regular
Исходный контур.

### Italic
Курсив получается аффинным сдвигом:

$$
x' = x + \alpha y, \qquad y'=y.
$$

### Bold
Для учебной демонстрации используется приближённая жирность: контур расширяется относительно центра буквы. Это не полноценный типографский offset, но хорошо показывает геометрическую идею утолщения.

### Custom
Авторская деформация: добавляется волнообразный сдвиг по оси `x`, зависящий от `y`. Такая деформация меняет локальную кривизну контура.


In [ ]:

# TODO: реализуйте стилевые преобразования
# regular — исходный контур
# italic — x' = x + alpha * y
# bold — учебное расширение контура
# custom — авторская деформация

def transform_segments(contours, transform_fn):
    raise NotImplementedError


def italic_style(contours, alpha=0.25):
    raise NotImplementedError


def bold_style(contours, factor=1.10):
    raise NotImplementedError


def custom_style(contours, amplitude=0.08, frequency=2.0):
    raise NotImplementedError


def styles_for_letter(letter):
    raise NotImplementedError



## 9) Галерея стилевых вариантов для пяти букв

Каждая строка — одна буква, каждый столбец — один стиль.


In [ ]:
# TO DO


## 10) Исследование кривизны

Для дискретно заданной кривой используем численную оценку кривизны:

$$
\kappa(t) \approx
\frac{|x'(t)y''(t)-y'(t)x''(t)|}
{(x'(t)^2+y'(t)^2)^{3/2}}.
$$

Ниже сравнивается кривизна исходного и авторски деформированного варианта одной буквы.


In [ ]:

# TODO: реализуйте численную оценку кривизны и сравните regular/custom

def discrete_curvature(points):
    raise NotImplementedError



## 11) Переход к 3D: экструзия буквы

Для одной выбранной буквы строится 3D-форма:

1. передний контур располагается в плоскости $z=0$;
2. задний контур располагается в плоскости $z=h$;
3. соответствующие точки переднего и заднего контуров соединяются боковыми поверхностями.

Для 3D-визуализации рекомендуется использовать **Plotly**, так как интерактивный график можно вращать.


**Если у вас будет желение - не ограничивайтесь одной буквой, можете сделать разные буквы и стили*

In [ ]:

# TODO: реализуйте 3D-экструзию буквы.
# Для 3D рекомендуется использовать Plotly.

def contour_front_back(points2d, depth=0.25):
    raise NotImplementedError


def plot_letter_3d_plotly(letter, style='regular', depth=0.25):
    raise NotImplementedError



## 12) Один боковой сегмент как поверхность Безье

Если взять один кубический сегмент переднего контура и такой же сегмент заднего контура, между ними получается поверхность Безье степени $3 	imes 1$:

$$
S(u,v)=\sum_{i=0}^{3}\sum_{j=0}^{1} Q_{ij}B_i^3(u)B_j^1(v).
$$

Здесь параметр $u$ идёт вдоль кривой, а параметр $v$ — от передней грани к задней.


In [ ]:

# TODO: покажите один боковой сегмент как поверхность Безье степени 3 x 1.
# Нужно построить контрольную сетку 4 x 2 и визуализировать её в Plotly.

def side_patch_from_bezier_controls(seg2d, depth=0.25):
    raise NotImplementedError


def bezier_surface_point(grid, u, v):
    raise NotImplementedError


def sample_bezier_surface(grid, nu=35, nv=8):
    raise NotImplementedError
